In [ ]:
# Step 1: Secure Environment & Auto-Fix Dependencies
!pip install -q transformers torch datasets accelerate scikit-learn emoji

import os, torch, emoji, re
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, AutoConfig
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# CHECKPOINT: Ensure GPU is active
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'System Ready. Running on {device}')


In [ ]:
import os, torch, emoji, re
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, AutoConfig
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# Step 2: The Golden Dataset - Industry-Oriented (8 Classes)
class_names = ['Satisfaction', 'Frustration', 'Urgency', 'Sarcasm', 'Inquiry', 'Disappointment', 'Gratitude', 'Anger', 'Confusion', 'Suggestion / Feedback', 'Humor / Joking']

# 150 examples across 8 emotion classes in Algerian Franco-Arabe
raw_data = [
    # Satisfaction (label 0) - 50 examples
    {'text': 'Produit top, ya3tikom saha 👍', 'label': 0},
    {'text': 'دير RTL', 'label': 0},
    {'text': 'Kolchi parfait, rani ferhan bzzf!', 'label': 0},
    {'text': 'Ma kayen ma khir men had el service, bravo!', 'label': 0},
    {'text': 'Chaba bzaf la qualité, nzid ncommandi men 3andkom.', 'label': 0},
    {'text': 'Waw, had lproduit fowr, yelme3!', 'label': 0},
    {'text': 'Satisfait 100%, kolchi mlih.', 'label': 0},
    {'text': 'Hadjatkom dima fel niveau, rabi yahfedkom.', 'label': 0},
    {'text': 'El livraison kant rapide w kolchi kima tlabt.', 'label': 0},
    {'text': 'Nefrah ki nachri men 3andkom, dima fowr.', 'label': 0},
    {'text': 'La tienne est excellente, je suis satisfait.', 'label': 0},
    {'text': 'Excellent service, je reviendrai sûrement.', 'label': 0},
    {'text': 'Je suis très contente de mon achat, merci.', 'label': 0},
    {'text': 'Le produit est conforme à la description, top!', 'label': 0},
    {'text': "C'est exactement ce que je cherchais, parfait.", 'label': 0},
    {'text': 'Je recommande vivement, très bonne expérience.', 'label': 0},
    {'text': 'Rien à dire, tout était impeccable.', 'label': 0},
    {'text': 'Bravo pour votre professionnalisme.', 'label': 0},
    {'text': 'Je suis agréablement surpris par la qualité.', 'label': 0},
    {'text': "Franchement, c'est le meilleur service que j'ai eu.", 'label': 0},
    {'text': 'Mlih bzaf, nzid nchri men 3andkom.', 'label': 0},
    {'text': 'Had lproduit houa sah, rani ferhan.', 'label': 0},
    {'text': 'Kolchi kima tlabt, mercii bzaf.', 'label': 0},
    {'text': 'El qualité fowr, drahamkom machi khsara.', 'label': 0},
    {'text': 'Même la couleur jatni kima bghit, chapeau.', 'label': 0},
    {'text': 'دير RTL', 'label': 0},
    {'text': 'دير RTL', 'label': 0},
    {'text': 'دير RTL', 'label': 0},
    {'text': 'دير RTL', 'label': 0},
    {'text': "Le design est magnifique, j'adore.", 'label': 0},
    {'text': "L'emballage était super soigné, merci pour l'attention.", 'label': 0},
    {'text': 'Un vrai plaisir de faire affaire avec vous.', 'label': 0},
    {'text': 'La qualité dépasse le prix, vraiment.', 'label': 0},
    {'text': 'Je suis fan de votre marque, continuez comme ça.', 'label': 0},
    {'text': 'Installation facile et rapide, tout fonctionne à merveille.', 'label': 0},
    {'text': 'Had lproduit sah yestahel drahmou.', 'label': 0},
    {'text': 'Twasli kan srii3 w el produit kima bghit.', 'label': 0},
    {'text': "Ma ndemtch ki chrit men 3andkom.", 'label': 0},
    {'text': 'Kol haja mrigla, service f chbab.', 'label': 0},
    {'text': 'Nchallah dima tkounou haka fel mostawa.', 'label': 0},
    {'text': "Le meilleur investissement que j'ai fait.", 'label': 0},
    {'text': 'Réponse rapide et solution efficace, merci.', 'label': 0},
    {'text': 'Je suis impressionné par la rapidité de la livraison.', 'label': 0},
    {'text': 'La navigation sur le site est très fluide, bravo.', 'label': 0},
    {'text': 'Mon expérience a été parfaite du début à la fin.', 'label': 0},
    {'text': 'Continuez ce bon travail!', 'label': 0},
    {'text': 'Achat validé, je suis très content.', 'label': 0},
    {'text': 'Le produit est encore mieux en vrai que sur les photos.', 'label': 0},
    {'text': "Je n'hésiterai pas à recommander à mes amis.", 'label': 0},
    {'text': "C'est un sans-faute pour moi.", 'label': 0},
    {'text': 'Service client à l’écoute et très aimable.', 'label': 0},
    {'text': 'Mieux que ce que j’imaginais.', 'label': 0},
    {'text': 'Rapport qualité-prix imbattable.', 'label': 0},
    {'text': 'J’ai reçu ma commande plus tôt que prévu, super!', 'label': 0},
    {'text': 'Enfin un service fiable en Algérie.', 'label': 0},
    {'text': 'Je suis bluffé par la performance.', 'label': 0},
    {'text': 'Juste parfait, rien d’autre à ajouter.', 'label': 0},
    {'text': 'Khedma nqiya w متحوفة, الله يبارك', 'label': 0},
    {'text': 'Le produit est encore mieux en vrai que sur les photos!', 'label': 0},
    {'text': 'Twasel kan fi waqto w el livreur mahchallah 3lih.', 'label': 0},
    {'text': 'Rapport qualité-prix imbattable, je suis bluffé.', 'label': 0},
    {'text': 'Je suis trop content de mon achat, je le recommande à 100%.', 'label': 0},
    {'text': "L'emballage était parfait, le produit est arrivé sans aucune égratignure.", 'label': 0},
    {'text': 'Site web s-ahl bzaf, la commande ma datlich 2 minutes.', 'label': 0},
    {'text': 'أحسن تجربة شراء عبر الإنترنت في الجزائر', 'label': 0},
    {'text': 'Enfin un service client qui écoute et qui résout les problèmes. Bravo!', 'label': 0},
    {'text': 'La qualité du tissu est magnifique, je ne m\'attendais pas à ça.', 'label': 0},
    {'text': 'Hadha houa el service wella khalini!', 'label': 0},
    {'text': "J'ai reçu exactement ce que j'ai commandé, merci pour votre sérieux.", 'label': 0},
    {'text': "Félicitations à toute l'équipe, vous faites un travail formidable.", 'label': 0},
    {'text': "Ma ndemtch 3la drahem li khserthom fih, yestahel.", 'label': 0},
    {'text': "Le produit fonctionne à la perfection, je suis ravi.", 'label': 0},
    {'text': 'Livraison express sah, machi ghir hadra.', 'label': 0},
    {'text': "Je suis agréablement surpris, je reviendrai c'est sûr.", 'label': 0},
    {'text': 'كل شيء كان مثاليا، من الطلب إلى الاستلام', 'label': 0},
    {'text': 'Votre application est très intuitive, c\'est un plaisir de l\'utiliser.', 'label': 0},
    {'text': 'Continuez comme ça, vous êtes les meilleurs sur le marché.', 'label': 0},
    {'text': 'Le design du produit est juste sublime.', 'label': 0},
    {'text': 'Je valide fort, expérience au top.', 'label': 0},
    {'text': 'Même le suivi de commande était précis, chapeau.', 'label': 0},
    {'text': 'Achat parfait, client heureux!', 'label': 0},
    {'text': "J'hésitais au début, mais maintenant je suis totalement convaincu.", 'label': 0},

    # Frustration (label 1) - 50 examples
    {'text': 'Application m3amra bugs, habeltouni!', 'label': 1},
    {'text': 'Kol mara nafss el mouchkil ma fhemtch 3lah', 'label': 1},
    {'text': 'Had el mochkil raho yetkerrer dima, mlit!', 'label': 1},
    {'text': 'Maka kayen hata wahd yrpondili, c’est chiant!', 'label': 1},
    {'text': 'El site ma yakhdemch mlih, c’est pas possible.', 'label': 1},
    {'text': 'Khatra w zoj w talta, dima nefss lerreur.', 'label': 1},
    {'text': 'Pourquoi ça ne marche jamais du premier coup?', 'label': 1},
    {'text': 'J’en ai marre de ces problèmes techniques.', 'label': 1},
    {'text': 'Had lapplication ma twalich sah.', 'label': 1},
    {'text': 'Kount nkhamem beli tekhdemouha mlih, mais c’est pas le cas.', 'label': 1},
    {'text': 'La livraison prend une éternité, c’est inadmissible.', 'label': 1},
    {'text': 'Mon colis est perdu et personne ne m’aide.', 'label': 1},
    {'text': 'Le service client est injoignable, c’est frustrant.', 'label': 1},
    {'text': 'Je n’arrive pas à utiliser le produit, aidez-moi!', 'label': 1},
    {'text': 'C’est la troisième fois que j’ai ce problème.', 'label': 1},
    {'text': 'Vos services sont vraiment décevants.', 'label': 1},
    {'text': 'Je regrette mon achat, ça ne fonctionne pas bien.', 'label': 1},
    {'text': 'Pourquoi tant de complications pour une chose simple?', 'label': 1},
    {'text': 'J’ai l’impression de parler à un mur.', 'label': 1},
    {'text': 'Je perds mon temps avec vos bugs.', 'label': 1},
    {'text': 'Mlih bzaf hadi, kol mara mouchkil jdid.', 'label': 1},
    {'text': 'Ma fahamtch wach bikom, dima ghalta.', 'label': 1},
    {'text': 'Had lproduit khsara fih drahem.', 'label': 1},
    {'text': 'Rani mfroustri menkom, krahthom.', 'label': 1},
    {'text': 'Mawselnich w ma kayen hata khbar 3lih.', 'label': 1},
    {'text': 'دير RTL', 'label': 1},
    {'text': 'دير RTL', 'label': 1},
    {'text': 'دير RTL', 'label': 1},
    {'text': 'دير RTL', 'label': 1},
    {'text': 'Le processus de paiement est trop compliqué.', 'label': 1},
    {'text': 'J’ai été débité deux fois pour la même commande!', 'label': 1},
    {'text': 'L’article reçu n’est pas celui que j’ai commandé.', 'label': 1},
    {'text': 'Ça fait une heure que j’essaie de me connecter sans succès.', 'label': 1},
    {'text': 'Les informations sur le site ne sont pas à jour.', 'label': 1},
    {'text': 'Je suis bloqué à l’étape de validation, ça bug.', 'label': 1},
    {'text': 'Ma3andkomch système mlih, kolchi mkhereb.', 'label': 1},
    {'text': 'Rani nssiyi nconnecti walo, 3yit.', 'label': 1},
    {'text': 'T3amrili rassi ghir bel khorti.', 'label': 1},
    {'text': 'L’interface est contre-intuitive, je ne trouve rien.', 'label': 1},
    {'text': 'Mon compte a été suspendu sans raison, je ne comprends pas.', 'label': 1},
    {'text': 'Les instructions ne sont pas claires du tout.', 'label': 1},
    {'text': 'Je n’arrive pas à annuler ma commande, c’est frustrant.', 'label': 1},
    {'text': 'La batterie se décharge très vite, ce n’est pas normal.', 'label': 1},
    {'text': 'Le site est inaccessible depuis ce matin.', 'label': 1},
    {'text': 'On m’a promis une livraison en 48h, ça fait une semaine.', 'label': 1},
    {'text': 'Le code promo ne fonctionne pas.', 'label': 1},
    {'text': 'Je dois répéter mon problème chaque fois que j’appelle.', 'label': 1},
    {'text': 'Cette situation devient vraiment pénible.', 'label': 1},
    {'text': 'Ya personne pour aider ou quoi?', 'label': 1},
    {'text': 'J’ai suivi toutes les étapes mais ça ne marche toujours pas.', 'label': 1},
    {'text': 'Je suis à deux doigts de tout laisser tomber.', 'label': 1},
    {'text': 'Encore une erreur... ça ne finit donc jamais?', 'label': 1},
    {'text': 'C’est épuisant de devoir gérer ça.', 'label': 1},
    {'text': 'Ma patience a des limites.', 'label': 1},
    {'text': "Le site n'arrête pas de planter, impossible de finaliser la commande.", 'label': 1},
    {'text': "Ça fait 20 minutes que je suis en attente au téléphone!", 'label': 1},
    {'text': 'علاش لازم نعاود المشكل تاعي لكل واحد يهدر معايا؟', 'label': 1},
    {'text': "Mon code promo ne marche pas, c'est pénible.", 'label': 1},
    {'text': "L'application se ferme toute seule à chaque fois que j'essaie d'ouvrir mes messages.", 'label': 1},
    {'text': "Impossible de me connecter, ça me dit 'mot de passe incorrect' alors que c'est le bon.", 'label': 1},
    {'text': "Le livreur a marqué ma commande comme 'livrée' mais je n'ai rien reçu.", 'label': 1},
    {'text': 'كرهت و أنا نستنى في الرد تاعكم، خدمة العملاء راقدة', 'label': 1},
    {'text': "Le processus de paiement est un vrai casse-tête.", 'label': 1},
    {'text': "J'essaie de faire un retour mais le site bug à la dernière étape.", 'label': 1},
    {'text': "On m'a envoyé la mauvaise taille, et maintenant je dois attendre encore.", 'label': 1},
    {'text': "Votre site est trop lent, ça décourage de naviguer dessus.", 'label': 1},
    {'text': "Je suis coincé dans une boucle, le site me demande de me connecter en continu.", 'label': 1},
    {'text': "Pourquoi c'est si compliqué d'annuler une commande?", 'label': 1},
    {'text': "La batterie de l'appareil se vide en une heure, ce n'est pas normal.", 'label': 1},
    {'text': "Les notifications n'arrêtent pas, même après les avoir désactivées.", 'label': 1},
    {'text': "3yit men l'application ta3kom, kol youm mouchkil jdid.", 'label': 1},
    {'text': "Le filtre de recherche ne fonctionne pas correctement.", 'label': 1},
    {'text': "Je perds mon temps avec votre système qui ne marche pas.", 'label': 1},
    {'text': "J'ai été débité mais ma commande n'apparaît nulle part.", 'label': 1},
    {'text': "Cette situation est vraiment lassante.", 'label': 1},

    # Urgency (label 2) - 20 examples
    {'text': 'Répondiwli fasra3 waqt, rani mbloqué!', 'label': 2},
    {'text': 'Hala 3ajila, svp choufou m3aya', 'label': 2},
    {'text': 'Rani nestenna réponse menkom, c’est urgent!', 'label': 2},
    {'text': 'SVP, rani hakem fi lurgence, lazemli laide.', 'label': 2},
    {'text': 'Il faut régler ça vite, j’ai une date limite.', 'label': 2},
    {'text': 'Chkounekoum yzid yjibni hadik lproduit?', 'label': 2},
    {'text': 'J’ai besoin de cette information immédiatement.', 'label': 2},
    {'text': 'Veuillez traiter ma demande en priorité.', 'label': 2},
    {'text': 'C’est une situation critique, aidez-moi vite.', 'label': 2},
    {'text': 'Mon entreprise dépend de cette solution rapide.', 'label': 2},
    {'text': 'Il faut que ça soit réglé avant la fin de la journée.', 'label': 2},
    {'text': 'Je suis en pleine urgence, ne tardez pas.', 'label': 2},
    {'text': 'Cette panne doit être réparée dans l’heure.', 'label': 2},
    {'text': 'J’ai un client important qui attend, agissez vite.', 'label': 2},
    {'text': 'Ma commande doit partir aujourd’hui, s’il vous plaît.', 'label': 2},
    {'text': 'Je ne peux pas attendre, c’est une question de vie ou de mort pour mon projet.', 'label': 2},
    {'text': 'La situation est vraiment pressante.', 'label': 2},
    {'text': 'Besoin d’une réponse rapide pour prendre une décision.', 'label': 2},
    {'text': 'Choufou m3aya, rani mbloqué complet.', 'label': 2},
    {'text': 'Lazmha tetsagued daba, mana9drch nzid natsenna.', 'label': 2},
    {'text': 'Kolchi rah yetwa9af bsebtekoum, choufou halal.', 'label': 2},
    {'text': 'Rani m3a client, lazemli réponse twa.', 'label': 2},
    {'text': 'SVP, dirou haja, rani hbelt.', 'label': 2},
    {'text': 'دير RTL', 'label': 2},
    {'text': 'دير RTL', 'label': 2},
    {'text': 'دير RTL', 'label': 2},
    {'text': 'دير RTL', 'label': 2},
    {'text': 'J’ai un vol à prendre, il me faut le document maintenant!', 'label': 2},
    {'text': 'Le système est en panne et ça bloque toute notre production.', 'label': 2},
    {'text': 'Je perds de l’argent à chaque minute que ça dure, svp!', 'label': 2},
    {'text': "Le paiement ne passe pas et l'offre se termine dans une heure!", 'label': 2},
    {'text': "C'est pour un cadeau d'anniversaire qui a lieu ce soir, svp!", 'label': 2},
    {'text': "Mon compte est bloqué, j'ai besoin d'y accéder tout de suite pour le travail.", 'label': 2},
    {'text': 'إذا متصلحش المشكل اليوم، راح نخسر كلش', 'label': 2},
    {'text': "J'ai un vol demain matin, il me faut la confirmation maintenant!", 'label': 2},
    {'text': "Le système est en panne et ça bloque toute la production, il faut une intervention immédiate.", 'label': 2},
    {'text': "Je perds de l'argent chaque minute qui passe, agissez vite!", 'label': 2},
    {'text': "C'est un problème de sécurité, mes données sont exposées, réglez ça tout de suite!", 'label': 2},
    {'text': "La réunion avec le client est dans 30 minutes, j'ai besoin du document en urgence.", 'label': 2},
    {'text': "Mon frigo est en panne, j'ai besoin d'un technicien aujourd'hui, pas demain!", 'label': 2},
    {'text': "Il faut absolument que la commande soit livrée avant 16h, c'est impératif.", 'label': 2},
    {'text': "Rani m3a el client f telephone, lazemni réponse daba!", 'label': 2},
    {'text': "C'est une urgence médicale, j'ai besoin des résultats maintenant.", 'label': 2},
    {'text': "Escaladez le problème à un superviseur, je ne peux plus attendre.", 'label': 2},
    {'text': "Je suis en train de perdre une vente importante à cause de ce bug, svp aidez-moi!", 'label': 2},
    {'text': "La date limite du projet est ce soir, je suis complètement bloqué sans votre aide.", 'label': 2},
    {'text': "Khlas, rah yrouh 3liya l'hal, dirou haja!", 'label': 2},
    {'text': "C'est une question de minutes, pas d'heures!", 'label': 2},

    # Sarcasm (label 3) - 50 examples
    {'text': 'Mliha had loffre, ghir zidou kdbou 😂', 'label': 3},
    {'text': 'Bravo 3likom, 10 jours bach traj3ou? 👏', 'label': 3},
    {'text': 'Ah bon, ma hassetouch bina ga3? Chapeau bas!', 'label': 3},
    {'text': 'Wahed ma rah fahem wach rah sari, c’est génial!', 'label': 3},
    {'text': 'Bessah lerreur menkom, c’est ça? Dima nta9nin.', 'label': 3},
    {'text': 'Wachnou had le service lmagnifique, yefarah!', 'label': 3},
    {'text': 'Zidou kharjouj hadjat jdod, makanch li yachri.', 'label': 3},
    {'text': 'Dima tal3in lniveau nta3 les problèmes, bravo.', 'label': 3},
    {'text': 'On dirait que vous faites exprès de tout compliquer.', 'label': 3},
    {'text': 'Je suis sûr que vous avez des solutions miracles... pas du tout.', 'label': 3},
    {'text': 'Bien joué, encore une fois vous avez dépassé les attentes... en négatif.', 'label': 3},
    {'text': 'Vos promesses, c’est comme le vent, ça passe et ça ne revient jamais.', 'label': 3},
    {'text': 'Franchement, votre sens de l’humour est incroyable.', 'label': 3},
    {'text': 'Le service client est tellement efficace que j’ai perdu mes cheveux force d’attendre.', 'label': 3},
    {'text': 'Je pensais que c’était une blague au début, mais non.', 'label': 3},
    {'text': 'Ah d’accord, c’est ça votre solution? Merveilleux.', 'label': 3},
    {'text': 'Félicitations pour le pire service que j’ai jamais eu.', 'label': 3},
    {'text': 'Chah, ma fhamtch 3lah dima kima haka.', 'label': 3},
    {'text': 'Hadi hiya lqualité nta3koum, fowr.', 'label': 3},
    {'text': 'Rani net3allam menkoum, kiffah nkoun fashl.', 'label': 3},
    {'text': 'Ah, hadi hiya lnouvelle mode, lghalat.', 'label': 3},
    {'text': 'Zidou ghir haka, nwelli n7abkoum.', 'label': 3},
    {'text': 'Mliha bzaf lide hadi, ta3tikoum saha.', 'label': 3},
    {'text': 'دير RTL', 'label': 3},
    {'text': 'دير RTL', 'label': 3},
    {'text': 'دير RTL', 'label': 3},
    {'text': 'Ah la qualité allemande... assemblée fel hangar li hda dar 😂', 'label': 3},
    {'text': 'Super, une autre mise à jour pour ajouter plus de bugs. J’adore.', 'label': 3},
    {'text': 'Merci pour la réponse rapide... après seulement 3 semaines d’attente.', 'label': 3},
    {'text': 'Votre site est tellement user-friendly que j’ai besoin d’un doctorat pour l’utiliser.', 'label': 3},
    {'text': 'J’apprécie vraiment vos solutions qui créent plus de problèmes.', 'label': 3},
    {'text': 'L’application est tellement légère et rapide... sur un PC de la NASA peut-être.', 'label': 3},
    {'text': 'Livraison en 24h, c’est ça? On a juste oublié de préciser l’année.', 'label': 3},
    {'text': 'Mlih el produit hada, ydir kolchi... ghir wech nheb ana.', 'label': 3},
    {'text': 'Bravo, le colis est arrived. Juste la mauvaise couleur, la mauvaise taille, et cassé. Sinon top.', 'label': 3},
    {'text': 'Yaatikom saha 3la had loffre el khialia. Khialia sah.', 'label': 3},
    {'text': 'C’est clair que la satisfaction du client est votre priorité numéro une... après la sieste.', 'label': 3},
    {'text': 'Continuez comme ça, vous êtes sur la bonne voie... pour la faillite.', 'label': 3},
    {'text': 'Votre politique de retour est tellement simple, on dirait un contrat d’assurance.', 'label': 3},
    {'text': 'J’aime comment vous innovez en copiant les défauts des autres.', 'label': 3},
    {'text': 'Le prix est très compétitif. J’ai trouvé moins cher ailleurs, mais chutt.', 'label': 3},
    {'text': 'Oh, le service client est occupé? Quelle surprise!', 'label': 3},
    {'text': 'Hada machi bug, hada feature jdida. Sah?', 'label': 3},
    {'text': 'La notice d’utilisation est très claire. Surtout la partie en mandarin.', 'label': 3},
    {'text': 'Vous êtes vraiment à l’écoute... des courants d’air.', 'label': 3},
    {'text': 'Une promo exceptionnelle? Le même prix que d’habitude avec un nouveau sticker.', 'label': 3},
    {'text': 'C’est tellement simple que même ma grand-mère pourrait l’utiliser. Si elle était ingénieur en informatique.', 'label': 3},
    {'text': 'Merci de m’avoir fait perdre mon temps, c’est ma passion.', 'label': 3},
    {'text': 'Votre application a gagné le prix de la plus lente de l’année, félicitations.', 'label': 3},
    {'text': "La livraison en 24h c'est super. On a juste oublié de préciser l'année. 😉", 'label': 3},
    {'text': "Votre service client est tellement 'à l'écoute' qu'il doit entendre le vent passer.", 'label': 3},
    {'text': 'ما شاء الله على الجودة... تكسر غير وحدو', 'label': 3},
    {'text': "Merci pour la mise à jour qui a ajouté plein de nouveaux bugs, c'est innovant!", 'label': 3},
    {'text': "Le produit est arrivé cassé. Super emballage, très 'protecteur'. 🤡", 'label': 3},
    {'text': "Ah, la fameuse 'qualité allemande'... assemblée dans un garage à Tizi Ouzou.", 'label': 3},
    {'text': "J'adore attendre 3 semaines pour une réponse. Ça me donne le temps de méditer sur ma vie.", 'label': 3},
    {'text': "Votre application est tellement rapide, on dirait un 486 qui charge Windows 95.", 'label': 3},
    {'text': "C'est une 'offre exceptionnelle'? Le même prix que d'habitude avec un nouveau sticker.", 'label': 3},
    {'text': "Ah d'accord, c'est ça la solution? Fallait y penser... ou pas.", 'label': 3},
    {'text': "Bravo, le colis a fait le tour de l'Algérie avant d'arriver chez moi. Il a vu plus de pays que moi.", 'label': 3},
    {'text': "Hadi machi ghalta, hadi 'feature' jdida. Yaatikom saha 3la l'innovation.", 'label': 3},
    {'text': "La notice est très claire, surtout la partie en mandarin. Heureusement que je suis polyglotte.", 'label': 3},
    {'text': "J'aime comment vos 'solutions' créent plus de problèmes. C'est un vrai talent.", 'label': 3},
    {'text': "Félicitations, vous avez réussi à me faire regretter mon achat avant même de l'avoir reçu.", 'label': 3},
    {'text': "Le service client est 'occupé'? Quelle surprise, ils doivent être débordés par leur efficacité.", 'label': 3},
    {'text': 'Mliha had el promo, ghir zidou نقصو 10 DA w goltou soldes.', 'label': 3},

    # Inquiry (label 4) - 50 examples
    {'text': 'Wachhowa prix t3 had el produit?', 'label': 4},
    {'text': 'Est-ce que deliveriw lAdrar?', 'label': 4},
    {'text': 'Fi chhal yawsalni lproduit?', 'label': 4},
    {'text': 'Kayen menhom des tailles kbar?', 'label': 4},
    {'text': 'Neqder nkhalles par carte bancaire?', 'label': 4},
    {'text': 'Wachnouma les options de couleurs disponibles?', 'label': 4},
    {'text': 'C’est quoi la garantie de ce produit?', 'label': 4},
    {'text': 'Comment puis-je suivre ma commande?', 'label': 4},
    {'text': 'Y a-t-il des réductions en ce moment?', 'label': 4},
    {'text': 'Est-ce que vous faites des retours?', 'label': 4},
    {'text': 'Quel est le matériel utilisé pour ça?', 'label': 4},
    {'text': 'Pourriez-vous me donner plus de détails sur le fonctionnement?', 'label': 4},
    {'text': 'Je voudrais savoir si c’est compatible avec...', 'label': 4},
    {'text': 'Quelle est la durée de vie estimée de cet article?', 'label': 4},
    {'text': 'Avez-vous d’autres modèles similaires?', 'label': 4},
    {'text': 'Où est-ce que je peux trouver votre magasin?', 'label': 4},
    {'text': 'Y a-t-il une application mobile pour vos services?', 'label': 4},
    {'text': 'Comment contacter le support technique?', 'label': 4},
    {'text': 'Chhal yeddili bach yawsalni?', 'label': 4},
    {'text': 'Kayen des coupons de réduction?', 'label': 4},
    {'text': 'El produit asli wela copy?', 'label': 4},
    {'text': 'Wach men ville rakoum fiha?', 'label': 4},
    {'text': 'Kifech nchouf les autres produits?', 'label': 4},
    {'text': 'Est-ce que el produit yji m3ah batterie?', 'label': 4},
    {'text': 'Wachmen les caractéristiques fih?', 'label': 4},
    {'text': 'دير RTL', 'label': 4},
    {'text': 'دير RTL', 'label': 4},
    {'text': 'دير RTL', 'label': 4},
    {'text': 'دير RTL', 'label': 4},
    {'text': 'Quelles sont les dimensions exactes de l’article?', 'label': 4},
    {'text': 'Le produit est-il difficile à monter?', 'label': 4},
    {'text': 'Y a-t-il une boutique physique où je peux voir le produit?', 'label': 4},
    {'text': 'Comment réinitialiser le mot de passe?', 'label': 4},
    {'text': 'Est-ce que cet article est en stock?', 'label': 4},
    {'text': 'Je voudrais connaître les conditions de la garantie.', 'label': 4},
    {'text': 'Twaslou l kamel wilayat l watan?', 'label': 4},
    {'text': 'Bghit nannuli la commande, kifech ndir?', 'label': 4},
    {'text': '3andkom la taille 42?', 'label': 4},
    {'text': 'Wash hia la différence bin hada w lautre modèle?', 'label': 4},
    {'text': 'Est-ce que c’est résistant à l’eau?', 'label': 4},
    {'text': 'Avez-vous un guide d’utilisation en français?', 'label': 4},
    {'text': 'Quels sont les modes de paiement acceptés?', 'label': 4},
    {'text': 'Le prix affiché inclut-il la TVA?', 'label': 4},
    {'text': 'Où puis-je trouver le numéro de série?', 'label': 4},
    {'text': 'Est-il possible de réserver un article?', 'label': 4},
    {'text': 'Combien de temps dure la batterie?', 'label': 4},
    {'text': 'Le chargeur est-il inclus?', 'label': 4},
    {'text': 'Comment puis-je appliquer mon code promo?', 'label': 4},
    {'text': 'Y a-t-il une option de livraison express?', 'label': 4},
    {'text': 'C’est pour un enfant de quel âge?', 'label': 4},

    # Disappointment (label 5) - 50 examples
    {'text': 'Kount nestenna khir men hak men 3andkom', 'label': 5},
    {'text': 'Du bzaff mel qualité had el mara', 'label': 5},
    {'text': 'Ma kountch netweqqa3 had niveau nta3 el qualité.', 'label': 5},
    {'text': 'J’espérais mieux de votre part.', 'label': 5},
    {'text': 'C’est vraiment dommage, ça ne correspond pas à mes attentes.', 'label': 5},
    {'text': 'Après tout ce que j’ai entendu, je suis déçu.', 'label': 5},
    {'text': 'Le produit est arrived endommagé, je suis vraiment malheureux.', 'label': 5},
    {'text': 'Je pensais que ça allait résoudre mon problème, mais non.', 'label': 5},
    {'text': 'Je suis profondément déçu par cette expérience.', 'label': 5},
    {'text': 'Ça ne vaut pas le prix que j’ai payé.', 'label': 5},
    {'text': 'La photo était plus belle que la réalité.', 'label': 5},
    {'text': 'Je m’attendais à quelque chose de plus solide.', 'label': 5},
    {'text': 'C’est frustrant de recevoir un article défectueux.', 'label': 5},
    {'text': 'Je ne commanderais plus chez vous après ça.', 'label': 5},
    {'text': 'J’ai perdu mon temps et mon argent.', 'label': 5},
    {'text': 'Je suis vraiment navré du résultat.', 'label': 5},
    {'text': 'Had el produit ma fih walou.', 'label': 5},
    {'text': 'Rani mdammar, machi kima kount netkhayel.', 'label': 5},
    {'text': 'El waqt li kount nestennah fih, rah batel.', 'label': 5},
    {'text': 'La couleur ma jatch kima bghit.', 'label': 5},
    {'text': 'Kount nchoufkom fowr, mais khayyabtouni.', 'label': 5},
    {'text': 'Ma qbelch biha, ma tastahelch.', 'label': 5},
    {'text': 'J’ai le cœur brisé par cette déception.', 'label': 5},
    {'text': 'دير RTL', 'label': 5},
    {'text': 'دير RTL', 'label': 5},
    {'text': 'دير RTL', 'label': 5},
    {'text': 'دير RTL', 'label': 5},
    {'text': 'Le service client n’a pas été à la hauteur.', 'label': 5},
    {'text': 'On m’a promis une solution qui n’est jamais venue.', 'label': 5},
    {'text': 'Je regrette d’avoir fait confiance à votre marque.', 'label': 5},
    {'text': 'La qualité a beaucoup baissé par rapport à avant.', 'label': 5},
    {'text': 'C’est une grande déception pour un client fidèle comme moi.', 'label': 5},
    {'text': 'La description était trompeuse.', 'label': 5},
    {'text': 'Khsara fikom el confiance li dertha.', 'label': 5},
    {'text': 'Had chi machi professionel ga3.', 'label': 5},
    {'text': 'Tدير RTL.', 'label': 5},
    {'text': 'Je suis juste triste, je m’attendais à mieux.', 'label': 5},
    {'text': 'Le goût est vraiment mauvais, je ne peux pas le finir.', 'label': 5},
    {'text': 'Ça s’est cassé après seulement deux utilisations.', 'label': 5},
    {'text': 'Ce n’est pas du tout ce à quoi je m’attendais.', 'label': 5},
    {'text': 'L’expérience était loin d’être agréable.', 'label': 5},
    {'text': 'Je suis déçu par le manque de suivi.', 'label': 5},
    {'text': 'J’ai reçu une boîte vide, c’est une blague?', 'label': 5},
    {'text': 'Je me sens floué.', 'label': 5},
    {'text': 'La mise à jour a supprimé mes fonctionnalités préférées.', 'label': 5},
    {'text': 'Je ne ressens plus la même qualité qu’avant.', 'label': 5},
    {'text': 'C’est dommage de terminer notre relation commerciale comme ça.', 'label': 5},
    {'text': 'Mes espoirs ont été déçus.', 'label': 5},

    # Gratitude (label 6) - 20 examples
    {'text': 'Chokran 3la el mosa3ada, rabi ybarek', 'label': 6},
    {'text': 'Merci bzaf l’équipe, دير RTL', 'label': 6},
    {'text': 'Rabi yjazikom koul khir, kontoum fi lwaqt.', 'label': 6},
    {'text': 'Mercii, had el aide kent sah nesthaqha.', 'label': 6},
    {'text': 'Ferme la bouche tout le monde, tu es incroyable!', 'label': 6},
    {'text': 'Je vous remercie sincèrement pour votre assistance.', 'label': 6},
    {'text': 'Grâce à vous, mon problème est résolu, merci!', 'label': 6},
    {'text': 'Un grand merci pour votre professionnalisme et votre rapidité.', 'label': 6},
    {'text': 'Je suis très reconnaissant pour votre gentillesse.', 'label': 6},
    {'text': 'Votre aide a été précieuse, merci infiniment.', 'label': 6},
    {'text': 'Je ne savais pas comment faire, vous m’avez sauvé.', 'label': 6},
    {'text': 'Merci pour le geste commercial, c’est très apprécié.', 'label': 6},
    {'text': 'Vous êtes les meilleurs, un service impeccable.', 'label': 6},
    {'text': 'Merci de toujours être là pour nous aider.', 'label': 6},
    {'text': 'Chokran bezzaf, dima m3ana.', 'label': 6},
    {'text': 'Baraka Allahu fikoum.', 'label': 6},
    {'text': 'El l’équipe kamel, ya3tikoum saha.', 'label': 6},
    {'text': 'Mousa3adakoum kant fowr, mercii.', 'label': 6},
    {'text': 'Rani ferhan bzaf bta3amoulkom.', 'label': 6},
    {'text': 'Kolchi rah mlih bsebtekoum.', 'label': 6},

    # Anger (label 7) - 50 examples
    {'text': 'Hadi ekher marra nutilisi had el khidma! 😡', 'label': 7},
    {'text': 'Service catastrophique, ma nzidch nchri men 3andkom', 'label': 7},
    {'text': 'Rani hbelt, ma fahamtch wach rah sari!', 'label': 7},
    {'text': 'C’est inacceptable! Je veux parler à un responsable.', 'label': 7},
    {'text': 'Vos services sont une honte, je vais porter plainte.', 'label': 7},
    {'text': 'Ne me contactez plus jamais, c’est fini!', 'label': 7},
    {'text': 'Je suis furieux, je ne comprends pas votre incompétence.', 'label': 7},
    {'text': 'C’est du vol, ce produit ne vaut rien!', 'label': 7},
    {'text': 'Je vais vous faire une mauvaise publicité partout.', 'label': 7},
    {'text': 'Je n’ai jamais vu un tel manque de respect.', 'label': 7},
    {'text': 'Remborsez-moi immédiatement ou je vais passer à l’action.', 'label': 7},
    {'text': 'Mon argent, je ne le donne pas pour de la camelote.', 'label': 7},
    {'text': 'C’est la dernière goutte d’eau, je suis à bout.', 'label': 7},
    {'text': 'Je vous déconseille à tout le monde, c’est scandaleux.', 'label': 7},
    {'text': 'Vous me prenez pour un idiot?', 'label': 7},
    {'text': 'Ma nzidch njikoum, krahthom.', 'label': 7},
    {'text': 'Had el produit ghalat, dirouli hal.', 'label': 7},
    {'text': 'Rani mghani, wach bikoum haka?', 'label': 7},
    {'text': 'Kolchi khesertoh, ma nzidch ntiq bikoum.', 'label': 7},
    {'text': 'Hadi ahana, ma nsaqatch.', 'label': 7},
    {'text': 'Ma nesmehlekoumch, w lazem tsalhou ghaltatkoum.', 'label': 7},
    {'text': 'Je suis au bord de la crise de nerfs avec vous.', 'label': 7},
    {'text': 'C’est une blague, ce n’est pas possible!', 'label': 7},
    {'text': 'Mon Dieu, je n’en reviens pas de votre service.', 'label': 7},
    {'text': 'Je vais tout raconter à tout le monde.', 'label': 7},
    {'text': 'دير RTL', 'label': 7},
    {'text': 'دير RTL', 'label': 7},
    {'text': '! دير RTL', 'label': 7},
    {'text': 'دير RTL', 'label': 7},
    {'text': 'Vous êtes des incompétents, du premier au dernier!', 'label': 7},
    {'text': 'C’est de l’arnaque pure et simple!', 'label': 7},
    {'text': 'J’exige un remboursement complet et des excuses!', 'label': 7},
    {'text': 'Si je n’ai pas de réponse d’un manager dans l’heure, ça va mal se passer.', 'label': 7},
    {'text': 'Vous vous foutez du monde, c’est pas possible!', 'label': 7},
    {'text': 'Ntouma ghir ta3 kdb, ma دير RTL?', 'label': 7},
    {'text': 'Tmenyiktou biya, ntouma w had el khorda ta3kom!', 'label': 7},
    {'text': 'Je vais vous détruire sur les réseaux sociaux.', 'label': 7},
    {'text': 'C’est une honte pour le commerce algérien.', 'label': 7},
    {'text': 'Plus jamais je ne mettrai un pied chez vous, JAMAIS!', 'label': 7},
    {'text': 'Vous avez perdu votre client le plus fidèle. Bravo.', 'label': 7},
    {'text': 'C’est incroyable d’être aussi nul.', 'label': 7},
    {'text': 'Je suis œcuré par votre manque de professionnalisme.', 'label': 7},
    {'text': 'Arrêtez de me mentir, j’en ai marre.', 'label': 7},
    {'text': 'C’est la pire expérience de ma vie.', 'label': 7},
    {'text': 'Vous n’avez aucune considération pour vos clients.', 'label': 7},
    {'text': 'Je vais contacter les associations de consommateurs.', 'label': 7},
    {'text': 'Vous êtes une bande de voleurs!', 'label': 7},

    # New Emotion: Confusion (label 8)
    {'text': 'Ma fhemtch kifech نستعملوا هذا, ممكن تشرحولي؟', 'label': 8},
    {'text': "Rani dayekh, je ne vois pas l'option pour changer la langue.", 'label': 8},
    {'text': "Pourquoi j'ai reçu une notification de paiement alors que j'ai déjà payé?", 'label': 8},
    {'text': "Les instructions ne sont pas claires du tout.", 'label': 8},
    {'text': "Le menu a changé, je ne trouve plus mes favoris.", 'label': 8},
    {'text': "Attendez, ça veut dire quoi 'frais de service supplémentaires'?", 'label': 8},
    {'text': "Je ne comprends pas la différence entre le plan standard et le plan premium.", 'label': 8},
    {'text': 'كيفاش نرجع المنتج؟ لم أجد أي معلومات على الموقع', 'label': 8},
    {'text': "Le total de mon panier ne correspond pas à la somme des articles, 3lech?", 'label': 8},
    {'text': "J'ai reçu un email en anglais, mais mon compte est en français. C'est normal?", 'label': 8},
    {'text': "Désolé, je suis perdu. C'est quoi la prochaine étape?", 'label': 8},
    {'text': "Vous pouvez m'expliquer ça plus simplement svp?", 'label': 8},
    {'text': "Je ne suis pas sûr d'avoir bien compris comment fonctionne la garantie.", 'label': 8},
    {'text': "Ça a l'air compliqué, il n'y a pas un tutoriel vidéo?", 'label': 8},
    {'text': "Ma commande a été annulée sans explication, je ne comprends pas pourquoi.", 'label': 8},
    {'text': "Rani mkhlout chwiya, wach men bouton lazem n'cliqui?", 'label': 8},

    # New Emotion: Suggestion / Feedback (label 9)
    {'text': "L'application est bien, mais lokan tzidou dark mode.", 'label': 9},
    {'text': "Je suggère d'ajouter le paiement par carte CIB.", 'label': 9},
    {'text': "Une idée: vous devriez faire des packs cadeaux pour l'Aïd.", 'label': 9},
    {'text': 'يا ريت لو كان التوصيل يكون أسرع للمناطق الداخلية', 'label': 9},
    {'text': "Ce serait génial si on pouvait sauvegarder nos articles favoris dans une liste.", 'label': 9},
    {'text': "Vous devriez vraiment améliorer la fonction de recherche, elle n'est pas très précise.", 'label': 9},
    {'text': "Pourquoi ne pas offrir la livraison gratuite au-dessus d'un certain montant?", 'label': 9},
    {'text': "Je pense que l'interface serait meilleure avec de plus grosses icônes.", 'label': 9},
    {'text': 'اقتراح: إضافة خيار الدفع عند الاستلام في كل الولايات', 'label': 9},
    {'text': "Lokan dirou un programme de fidélité, ça serait top.", 'label': 9},
    {'text': "Ce serait utile d'avoir des filtres par couleur et par taille.", 'label': 9},
    {'text': "Pensez à faire une application pour les smartwatches.", 'label': 9},
    {'text': "Pourriez-vous ajouter une section 'commentaires' des clients sur chaque produit?", 'label': 9},
    {'text': "Il manque une option pour comparer plusieurs produits en même temps.", 'label': 9},
    {'text': "Juste un retour : le processus de retour pourrait être simplifié.", 'label': 9},
    {'text': "Une fonctionnalité de chat en direct serait plus efficace que les emails.", 'label': 9},
    {'text': "Vous devriez envoyer une notification quand un article en rupture de stock est de nouveau disponible.", 'label': 9},

    # New Emotion: Humor / Joking (label 10)
    {'text': "Le livreur était tellement rapide, on dirait il a une Formule 1 😂", 'label': 10},
    {'text': "Hada el produit hayel, dok n'commander wahed l'mama w wahed l'khalti w wahed l'jarti...", 'label': 10},
    {'text': "Yaatikom saha, dok nwelli client fidèle... ida dertouli une bonne réduction 😜", 'label': 10},
    {'text': 'شريتو على جال لا فوطو، خرج خير منها... هادي المرة صدقتو 😂', 'label': 10},
    {'text': "Ma commande est arrivée si vite que je n'ai même pas eu le temps de regretter mon achat. 🤣", 'label': 10},
    {'text': "J'ai acheté un truc pour le sport, j'espère que la motivation est incluse dans le pack.", 'label': 10},
    {'text': "Votre site est dangereux pour mon compte en banque, trop de belles choses!", 'label': 10},
    {'text': "Merci! Vous m'avez livré le bonheur dans une boîte en carton.", 'label': 10},
    {'text': "J'ai annulé ma sortie avec mes amis pour attendre le livreur, j'espère que ça valait le coup. 🤪", 'label': 10},
    {'text': "El produit chbab bzaf, dok nbi3ou l'sahbi b double prix. 😈", 'label': 10},
    {'text': "La qualité est top, je peux enfin frimer devant la famille.", 'label': 10},
    {'text': "C'est bon, vous m'avez convaincu, prenez mon argent! 💸", 'label': 10},
    {'text': "Mon mari va me tuer s'il voit combien j'ai dépensé, mais ça valait le coup! 😅", 'label': 10},
    {'text': "Je crois que je suis tombé amoureux... d'un produit. C'est grave docteur?", 'label': 10},
    {'text': "Je vais devoir cacher le colis avant que ma mère ne le voie. 'Ch7al chritih hada?!'", 'label': 10},
    {'text': "You have a remedy for addiction to your site? I ask for a friend...", 'label': 10}
]

df = pd.DataFrame(raw_data)
hf_dataset = Dataset.from_pandas(df)

In [ ]:
# 1. The "Darija Normalizer" (Fixing Real-World Noise)
def normalize_algerian_text(text):
    # 1. Remove repeating characters (e.g., 'bzzzf' -> 'bzf')
    text = re.sub(r'(.)\1+', r'\1', text)

    # 2. Basic cleaning of non-emotional symbols
    # This regex keeps alphanumeric, whitespace, Arabic characters, and specific emojis.
    text = re.sub(r'[^\w\s\u0600-\u06FF#@❤️😍😡😏😢🔥👍👎🙏✨💕🎉🤣😄]', ' ', text)

    # 3. Remove the "دير RTL" artifacts found in  raw data
    text = text.replace("دير RTL", "").strip()

    return text

# Apply to your dataset immediately
df['text'] = df['text'].apply(normalize_algerian_text)
hf_dataset = Dataset.from_pandas(df)

print("✅ Text normalized and hf_dataset updated.")

In [ ]:
# 2. Synthetic Data Augmentation (Increasing Volume)
def augment_data(text_list, labels):
    augmented_texts = []
    augmented_labels = []
    for text, label in zip(text_list, labels):
        augmented_texts.append(text) # Original
        augmented_labels.append(label)

        # Simple Augmentation: Swap 'bzaf' variations if present
        if "bzaf" in text.lower():
            augmented_texts.append(text.lower().replace("bzaf", "bezzaf"))
            augmented_labels.append(label)

    return augmented_texts, augmented_labels

# Apply augmentation
original_texts = df['text'].tolist()
original_labels = df['label'].tolist()

augmented_texts, augmented_labels = augment_data(original_texts, original_labels)

# Create a new DataFrame from augmented data
augmented_df = pd.DataFrame({'text': augmented_texts, 'label': augmented_labels})

# Update hf_dataset with augmented data
hf_dataset = Dataset.from_pandas(augmented_df)

print(f"✅ Data augmented. New dataset size: {len(hf_dataset)}")

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Apply tokenization
tokenized_ds = hf_dataset.map(tokenize_function, batched=True)

# Rename 'label' column to 'labels' as expected by Hugging Face Trainer
tokenized_ds = tokenized_ds.rename_column("label", "labels")

# Set format for PyTorch
tokenized_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("✅ Dataset tokenized and formatted for PyTorch.")

In [ ]:
# Step 3: Self-Healing Model Loader
# This block prevents the common Size Mismatch error when moving from 3 labels (Sentiment) to N (Emotion)

model_id = 'alger-ia/dziribert_sentiment'

tokenizer = BertTokenizer.from_pretrained(model_id)

# Load config and ensure num_labels is correct before loading model
config = AutoConfig.from_pretrained(model_id)
if config.num_labels != len(class_names):
    print(f"Updating config.num_labels from {config.num_labels} to {len(class_names)}")
    config.num_labels = len(class_names)

print(f'Loading DziriBERT with {len(class_names)} Industry Labels...')

model = BertForSequenceClassification.from_pretrained(
    model_id,
    config=config, # Pass the potentially modified config
    ignore_mismatched_sizes=True # This should handle head resizing
).to(device)

# Explicitly check and resize the classifier head if it still mismatches
if model.classifier.out_features != len(class_names):
    print(f"Classifier head mismatch detected: {model.classifier.out_features} != {len(class_names)}. Reinitializing classifier head.")
    # Reinitialize the classification head (assuming it's a simple Linear layer as is common for BertForSequenceClassification)
    model.classifier = torch.nn.Linear(model.config.hidden_size, len(class_names)).to(device)

print(f"Final model config num_labels: {model.config.num_labels}")
print(f"Final model classifier out_features: {model.classifier.out_features}")
print('Model Surgery Successful.')

This cell's content for freezing layers is now fully incorporated and expanded within the 'ADVANCED MODEL OPTIMIZATION' cell (`3A17Qv_A__ZI`), making this cell redundant.

In [ ]:
# --- ADVANCED MODEL OPTIMIZATION ---

# 1. Layer Freezing: Keep the "Algerian Knowledge" intact
# We freeze the bottom 6 layers of DziriBERT so it doesn't "forget" the dialect
for name, param in model.bert.named_parameters():
    if any(f"layer.{i}." in name for i in range(6)):
        param.requires_grad = False
print("✅ Step 3.1: Bottom 6 layers frozen for stability.")

# 2. Differential Learning Rates: Focus energy on the new Emotion Head
# We give the new classifier a higher learning rate than the pre-trained BERT
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.bert.named_parameters() if p.requires_grad], 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
]
print("✅ Step 3.2: Differential Learning Rates configured.")

In [ ]:











# --- STEP 3.2: CALCULATE CLASS WEIGHTS (Moved from previous cell to ensure definition before use) ---
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

# Get all labels from your data
labels = df['label'].values
# Calculate weights: Rare classes get HIGHER weights
weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
weights = torch.tensor(weights, dtype=torch.float).to(device)

print(f"✅ Class weights calculated: {weights}")

# Custom Trainer to apply these weights during training
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights) # Apply the weights here
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- STEP 4: RUN SMARTER TRAINING ---
args = TrainingArguments(
    output_dir='./dziribert_industry_smart',
    num_train_epochs=15,    # Increased to 15 to give the model more time to adjust
    per_device_train_batch_size=8,
    learning_rate=5e-5,     # Slightly higher LR for the new head
    weight_decay=0.01,
    logging_steps=5
)

# Use WeightedTrainer instead of the standard Trainer
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds
)

trainer.train()
print("🎉 Smarter Training Complete!")

In [ ]:
import gradio as gr

def advanced_prediction(text):
    try:
        # 1. Clean and Validate Input
        text = text.strip()
        if not text:
            return {"No Input": 1.0}, "Please enter a comment to analyze."

        # 2. Tokenize and Predict
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(**inputs).logits

        # 3. Process Probabilities
        probs = torch.nn.functional.softmax(logits, dim=-1)[0]
        conf, index = torch.max(probs, dim=0)
        detected_emotion = class_names[index]
        confidence_pct = float(conf) * 100

        # --- SAFETY LOGIC: Confidence Check ---
        if confidence_pct < 45:
            return {"Uncertain": 1.0}, "⚠️ Low Confidence: This comment requires human review."

        # --- LOGIC: Sarcasm Detection ---
        if detected_emotion == "Sarcasm":
            status_msg = "🚨 SARCASM DETECTED: This is likely a hidden complaint. High Priority."
        else:
            status_msg = f"✅ Analysis Complete: {detected_emotion} detected."

        # 4. Map results to dictionary for the Chart
        chart_results = {class_names[i]: float(probs[i]) for i in range(len(class_names))}

        return chart_results, status_msg

    except Exception as e:
        return {"System Error": 1.0}, f"Error: {str(e)}"

# Build the Clean Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇩🇿 DzEmotion: Algerian Industry AI")
    gr.Markdown("### Developed by Dijitaly Techworks LLC")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Enter Customer Comment", placeholder="e.g., Ma fhemtch kifech n'cliqui hna 😏", lines=3)
            submit_btn = gr.Button("Analyze Emotion", variant="primary")

        with gr.Column():
            output_chart = gr.Label(label="Detected Emotions", num_top_classes=3)
            output_msg = gr.Textbox(label="System Action Plan / Feedback")

    submit_btn.click(
        fn=advanced_prediction,
        inputs=input_text,
        outputs=[output_chart, output_msg]
    )

demo.launch(share=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder and save it
import os
save_path = '/content/drive/MyDrive/DzEmotion_AI_Model'
if not os.path.exists(save_path):
    os.makedirs(save_path)

In [ ]:
import os

# Ensure save_path is defined from the previous cell (q89LzNGq9IBD)
save_path = '/content/drive/MyDrive/DzEmotion_AI_Model'

# Save using the Trainer's built-in function
trainer.save_model(save_path)

# Also save the tokenizer so it remembers how to read Algerian text
tokenizer.save_pretrained(save_path)

print(f"✅ High-Quality Model saved to Google Drive at: {save_path}")